# 📦 Packages Structure — The Family Tree of Files in the Cupboard

A single module solved the “reuse your functions” problem. But real projects grow large: hundreds of modules, thousands of functions. Dumped in one flat folder, that becomes chaos. Python's answer is a **package** — a folder of related modules with an `__init__.py` in it, forming a tidy family tree that mirrors how real projects are organised.

## 1. What you will learn in this notebook

1. **Package = folder + `__init__.py`** — the marker that turns an ordinary directory into an importable family of modules.
2. **Three import styles for packages** — `import pkg`, `from pkg import mod`, and dotted imports like `pkg.mod.func`.
3. **The power of `__init__.py`** — a package banner, plus *re-export shortcuts* so users can reach deep functions with a short name.
4. **Relative imports** — how submodules talk to each other inside the package with leading-dot paths like `from .billing import tax`.
5. **The dunder zoo** — `__name__`, `__doc__`, `__package__`, `__path__`, and the `__pycache__` folder, i.e. how a package identifies and presents itself.

## 2. The big idea — order in the house-cupboard

Picture your kitchen storage. Everything mixed in one drawer is a hotchpotch; labelled boxes — kitchen, study, tools — make every item predictable. A Python project is the same: instead of fifty loose files, we group them into packages. `dukan/` becomes a shop with a `billing` shelf and a `catalog` shelf; `study/` becomes a room with a `notes` notebook. The `__init__.py` in each folder is the index-card that says what that drawer contains and which shortcuts it offers.

## 3. How the demo works

All demos build the package entirely in a temporary folder using `tempfile` and plain file writes, then add the parent folder to `sys.path`. Since `__init__.py` executes on the very first import, you will hear the package announce itself with a banner — proof that loading a package is not free silence but a deliberate, scriptable event. Files are written exactly as they would appear in a real project, so everything you learn here transfers directly to a repository on disk.

## 🗄️ Real-World Hook: Order in the House-Cupboard

Everything mixed in one drawer becomes a hotchpotch you can never find anything in. The organised alternative is **labelled boxes**: each box holds related items, and the outside of the box says what is inside.

```text
home/
├── kitchen/   ← a package! (__init__.py = the index-card)
│   ├── daal_shelf.py      ← a submodule
│   └── masala_rack.py     ← a submodule
└── study/     ← a package!
    └── notes.py
```

In Python, a folder becomes a package when it contains `__init__.py` — the index-card. That file can run setup code, print a banner, and re-export shortcuts so the family's most-used names are one step away.

### 📖 Definition (interview-grade)

> **Package** — a folder containing `__init__.py` and its submodules, importable by dotted name (`dukan.billing`). On the first import, `__init__.py` **executes**, doing the wiring and re-export work.
> (Modern Python also accepts a “namespace package” without `__init__.py` — but the explicit classic style is clearer and what the ecosystem mostly uses.)

# 1. The Package Build — A Folder, an `__init__.py`, and Three Submodules

## 1.1 Making a package with bare hands

The next cell assembles a complete package from nothing: an empty temp folder, then three files written with `open()`. The folder is named `dukan` (a little corner shop), and inside it sit the classic trio:

- `__init__.py` — the package's identity card. It carries the docstring, prints the welcome banner (“dukan-package banner: welcome ji! 🙏”), and performs the re-export `from .billing import total_bill`. That last line is the magic shortcut: after `import dukan`, callers can say `dukan.total_bill(...)` instead of digging into `dukan.billing.total_bill(...)`.
- `billing.py` — the money submodule. It defines a `TAX_RATE = 0.05` constant, a `tax(amount)` rounding 5% to two decimals, and `total_bill(amount)` returning amount plus tax.
- `catalog.py` — the price-board submodule. `ITEMS` maps dish names to prices, and `describe(item)` formats a dish as “name ₹price”.

## 1.2 The import mechanics

The parent folder — the one *containing* `dukan` — is inserted into `sys.path`, not the package itself. That is because a package is found by name: Python looks for a *folder* called `dukan` in one of the `sys.path` entries. The tree-print is just our own documentation of what was built, and then the real event happens: `import dukan` executes its `__init__.py`, and the banner rings out — the package has loaded.

In [1]:
# 🐣 STEP-1: Dukaan-package banao + import karo! (tempdir-full-tree! 🏪)
import sys, os, tempfile

BASE = tempfile.mkdtemp()
PKG = os.path.join(BASE, "dukan")
os.mkdir(PKG)

INIT_CODE = '''"""dukan — gali-ka-store family-package! 🏪"""
print("dukan-package banner: welcome ji! 🙏")

from .billing import total_bill          # SHORTCUT re-export! 📣
'''

BILLING_CODE = '''"""billing — tax+totals ka hisaab! 💰"""

TAX_RATE = 0.05

def tax(amount):
    """5% tax nikaalo amount pe!"""
    return round(amount * TAX_RATE, 2)

def total_bill(amount):
    """amount + tax ka grand-total!"""
    return round(amount + tax(amount), 2)
'''

CATALOG_CODE = '''"""catalog — items ke price-board! 📋"""

ITEMS = {"samosa": 20, "chai": 12, "jalebi": 80}

def describe(item):
    """item → 'name ₹price' format!"""
    return f"{item} ₹{ITEMS[item]}"
'''

with open(os.path.join(PKG, "__init__.py"), "w") as f:
    f.write(INIT_CODE)
with open(os.path.join(PKG, "billing.py"), "w") as f:
    f.write(BILLING_CODE)
with open(os.path.join(PKG, "catalog.py"), "w") as f:
    f.write(CATALOG_CODE)

# tree-darshan ASCII!
print("📂 package tree built:")
print("dukan/")
print("├── __init__.py   (banner + re-export 📣)")
print("├── billing.py    (tax/total_bill 💰)")
print("└── catalog.py    (ITEMS + describe 📋)")

sys.path.insert(0, BASE)                              # parent-folders path me!
import dukan                                          # 🔔 package-banner FIRES!
# Expected OUTPUT:
# 📂 package tree built:
# dukan/
# ├── __init__.py   (banner + re-export 📣)
# ├── billing.py    (tax/total_bill 💰)
# └── catalog.py    (ITEMS + describe 📋)
# dukan-package banner: welcome ji! 🙏

📂 package tree built:
dukan/
├── __init__.py   (banner + re-export 📣)
├── billing.py    (tax/total_bill 💰)
└── catalog.py    (ITEMS + describe 📋)
dukan-package banner: welcome ji! 🙏


## 1.3 Three import styles for packages

Packages multiply the import options you already know:

**Style A — package-level re-export.** Because `__init__.py` did `from .billing import total_bill`, the name `total_bill` became part of the package's top level. `dukan.total_bill(1000)` → `1050.0` (1000 + 5% tax) works with no submodule import at all. This is why `__init__.py` is called the shortcut-maker.

**Style B — dotted submodule import.** `from dukan.billing import tax` descends folder → file → name, then you call bare `tax(2000)` → `100.0`. The dotted path is explicit about *where* the function lives — great for readability and for linters to trace.

**Style C — module import.** `from dukan import catalog` grabs the whole submodule as a name, so you use `catalog.describe("samosa")` (→ `samosa ₹20`) and `catalog.ITEMS["jalebi"]` (→ `80`). This keeps related functions grouped under one object.

All three are the same data behind different doors. Choosing one is about legibility: shortcuts for hot public APIs, dotted imports for deep access, module imports when you want everything a submodule offers.

In [2]:
# 🛒 STEP-2: 3-import-styles for packages!
# ORDER-PATTERN-A: package-re-export shortcut call!
print("via __init__   :", dukan.total_bill(1000))     # from .billing re-export!

# ORDER-PATTERN-B: dotted-import submodule!
from dukan.billing import tax
print("dotted-import  :", tax(2000), "(5% of 2000!)")

# ORDER-PATTERN-C: from-package-import-module!
from dukan import catalog
print("module-import  :", catalog.describe("samosa"))
print("catalog-price  :", catalog.ITEMS["jalebi"])
# Expected OUTPUT:
# via __init__   : 1050.0
# dotted-import  : 100.0 (5% of 2000!)
# module-import  : samosa ₹20
# catalog-price  : 80

via __init__   : 1050.0
dotted-import  : 100.0 (5% of 2000!)
module-import  : samosa ₹20
catalog-price  : 80


# 2. Relative Imports — Wiring Inside the Package

## 2.1 Dots mean “inside this package”

Submodules often need each other. How does `billing.py` reach the price board in `catalog.py`? Through a **relative import**: `from .catalog import ITEMS`. The single leading dot means “the package I live in”, so the path is resolved relative to the current module's package, not the project root.

That is what `__init__.py` did with `from .billing import ...` — the dot says “billing, my sibling, in my package dukan.” Relative imports keep package wiring portable: you can rename the outer project folder or move the whole package, and the internal connectors still work, because they are relative to the package, not absolute.

## 2.2 The one sharp rule: never run a submodule directly

Relative imports work only when the file is loaded *as part of its package* — i.e. reached through an import like `import dukan` or `from dukan import billing`. If you try `python dukan/billing.py`, Python treats `billing.py` as a standalone script with no enclosing package, the leading dot has nothing to resolve, and you get `ImportError: attempted relative import with no known parent package`.

The practical consequence is the golden layout: **entry files (the ones you run) sit in the root folder; submodules are only imported, never executed directly.** The cell below states this as a set of rules to carry into your own projects.

In [3]:
# 🔌 STEP-3: Relative-imports — package-internal wiring ka note! 🎯
# billing.py ne catalog-se-something chahiye hota toh:
#     from .catalog import ITEMS        ← SAME-package sibling!
# Dot-ka-matlab: "current-package inside". Direct-run pe relative FAIL:
#     python billing.py  → ImportError: attempted relative import...
# > packages ke-subs script-run-mat-karo; entry-file root-folder me rakho!

print("RELATIVE-IMPORT GYAAN:")
print("  billing.py → `from .catalog import ITEMS` ✅ (sibling-link!)")
print("  root-run   → `python dukan/billing.py` ❌ (relative-na-mile!)")
print("  entry-file → root folder me `main.py` rakho ✅")
# Expected OUTPUT:
# RELATIVE-IMPORT GYAAN:
#   billing.py → `from .catalog import ITEMS` ✅ (sibling-link!)
#   root-run   → `python dukan/billing.py` ❌ (relative-na-mile!)
#   entry-file → root folder me `main.py` rakho ✅

RELATIVE-IMPORT GYAAN:
  billing.py → `from .catalog import ITEMS` ✅ (sibling-link!)
  root-run   → `python dukan/billing.py` ❌ (relative-na-mile!)
  entry-file → root folder me `main.py` rakho ✅


# 3. The Dunder Zoo — How a Package Describes Itself

## 3.1 Reading a package's identity card

The last live demo asks `dukan` to reveal itself through its **dunder attributes** — the double-underscore metadata every module and package carries:

- `dukan.__name__` → `"dukan"` — the package's registered import name.
- `dukan.__doc__` → the docstring written in `__init__.py` — its one-line biography.
- `dukan.__package__` → `"dukan"` — the package a module belongs to (a top-level package is its own parent).
- `dukan.__path__` → the list of directories where Python will hunt for submodules. This is the defining trait of a package: *only* packages have `__path__`; plain modules do not. `len(dukan.__path__) == 1` prints `True`, confirming exactly one folder backs this package.

These four attributes are the fastest way to interview an unfamiliar package: who are you, what do you promise, and where do your children live? Combined with the earlier `__pycache__` mention, you now see the whole lifecycle: source files → bytecode cache → loaded package object → queryable dunders. The final cleanup (`sys.path.pop(0)` and `sys.modules.pop("dukan", None)`) restores the import system for the next notebook.

In [4]:
# 🧰 STEP-4: Dunder-zoo peep (package-vals!) 🔍
print("dukan.__name__   :", dukan.__name__)
print("dukan.__doc__    :", dukan.__doc__)
print("dukan.__package__:", dukan.__package__)
print("dukan.__path__   : [folder-check]", len(dukan.__path__) == 1)

# cache-clean-tyag (demo KEEP-safe!)
sys.path.pop(0); sys.modules.pop("dukan", None)
# expected-na:
# Expected OUTPUT:
# dukan.__name__   : dukan
# dukan.__doc__    : dukan — gali-ka-store family-package! 🏪
# dukan.__package__: dukan
# dukan.__path__   : [folder-check] True

dukan.__name__   : dukan
dukan.__doc__    : dukan — gali-ka-store family-package! 🏪
dukan.__package__: dukan
dukan.__path__   : [folder-check] True


<module 'dukan' from '/tmp/tmpkoradz4c/dukan/__init__.py'>

## ⚠️ 5 Common Mistakes

| # | Mistake | Correct habit |
|---|---|---|
| 1 | forgetting `__init__.py` (a plain folder, not a package!) | add `__init__.py` — even a docstring-only one is enough 📁 |
| 2 | heavy compute inside `__init__.py` (slow import!) | keep init light: shortcuts and banners, not business logic 🐌 |
| 3 | running a submodule directly: `python dukan/billing.py` | keep entry files at the root; let the package be imported from above 🧭 |
| 4 | `from dukan import *` — pollution travels to packages too | use explicit names or dotted imports |
| 5 | package names in CAPITALS or with dashes | lowercase `snake_case` is the ecosystem standard 📁 |

> **🎯 Import-resolution map:** cache → path-scan → folder-peep → `__init__`-execute → dotted-binds. Understanding each step makes any package mystery solvable in minutes.

## Summary

A package is a folder of related modules marked by an `__init__.py` — the index-card that executes on first import, prints a banner, and re-exports shortcuts such as `from .billing import total_bill`. Packages can be imported three ways: the re-export shortcut (`dukan.total_bill(1000)`), the dotted submodule import (`from dukan.billing import tax`), and the whole-module import (`from dukan import catalog`). Submodules connect internally with relative imports (leading dots), which only work when the file is loaded as part of its package — so entry files live in the root folder, never run as `python dukan/billing.py`. Finally, the dunder quartet `__name__`, `__doc__`, `__package__`, and `__path__` (the true package marker) describe a package's identity, and `__pycache__` stores compiled bytecode. You are now ready to build your own package in the task notebook.